## 09_plotly

In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

conn = sqlite3.connect("../data/checking-logs.sqlite")
cur = conn.cursor()

In [2]:
checker = pd.read_sql(
    """
        SELECT uid, timestamp, numTrials
            FROM checker
            WHERE
                status = 'ready'
                AND uid LIKE 'user%'
                AND labname = 'project1'
            
            """,
    conn,
    parse_dates="timestamp",
)
checker["timestamp"] = checker.timestamp.dt.date
checker

,uid,timestamp,numTrials
0,user_4,2020-04-17,1
1,user_4,2020-04-17,2
2,user_4,2020-04-17,3
3,user_4,2020-04-17,4
4,user_4,2020-04-17,5
...,...,...,...
946,user_19,2020-05-15,26
947,user_19,2020-05-15,27
948,user_19,2020-05-15,28
949,user_28,2020-05-15,27


In [3]:
checker = checker.sort_values(by="timestamp")
df = checker.groupby(["timestamp", "uid"]).max()["numTrials"].unstack()
df.iloc[0] = df.iloc[0].fillna(0)
df = df.fillna(method="pad", axis=0)
df = df.transpose()
df

C:\Users\79611\AppData\Local\Temp\ipykernel_33832\1936923662.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="pad", axis=0)


timestamp,2020-04-17,2020-04-18,2020-04-19,2020-04-22,2020-04-23,2020-04-24,2020-05-03,2020-05-04,2020-05-05,2020-05-06,2020-05-07,2020-05-08,2020-05-09,2020-05-10,2020-05-11,2020-05-12,2020-05-13,2020-05-14,2020-05-15
uid,,,,,,,,,,,,,,,,,,,
user_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,11.0
user_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,21.0,59.0,59.0
user_11,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
user_12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0
user_13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,30.0,30.0,32.0,32.0
user_14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,18.0,25.0,49.0,92.0,92.0,99.0,99.0
user_15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,3.0
user_16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,3.0,10.0,10.0
user_17,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,3.0,6.0,6.0


In [4]:
initial_data = [
    go.Scatter(x=np.array([]), y=np.array([]), mode="lines+markers", name=name)
    for name in df.index
]
frames = [
    go.Frame(
        data=[
            go.Scatter(
                x=np.arange(1, i + 1),
                y=np.array(df.iloc[j, : i + 1]),
                mode="lines+markers",
                name=df.index[j],
            )
            for j in range(len(df.index))
        ]
    )
    for i in range(len(df.columns) + 1)
]
fig = go.Figure(
    data=initial_data,
    layout={
        "width": 1200,
        "height": 600,
        "title": "Dynamic of commits per user in project1",
        "xaxis": {"range": (0, len(df.columns) + 1), "dtick": 2},
        "yaxis": {"range": (0, round(max(df.max()) + 2, -1) + 1)},
        "updatemenus": [
            {
                "type": "buttons",
                "buttons": [{"method": "animate", "label": "play", "args": [None]}],
            }
        ],
    },
    frames=frames,
)
fig.show()

In [5]:
conn.close()